# 01 — Exploração e análise descritiva (EDA)

Estatísticas descritivas, visualizações e matriz de correlação.

## 1. Imports

In [ ]:
# Imports (ajustar conforme o dataset)
import time
import pandas as pd
from pysus import SINASC
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np

## 2. Conexão com o DATASUS / SINASC

Conecta à API do SINASC via `pysus` e inspeciona os grupos e metadados disponíveis.

In [ ]:
sinasc = SINASC().load() # Loads the files from DATASUS

In [ ]:
sinasc.metadata

In [ ]:
sinasc.groups

## 3. Download dos Dados

Baixa as Declarações de Nascidos Vivos (DN) do estado de **Minas Gerais** para os anos de **2020, 2021 e 2022** diretamente do DATASUS.
Os arquivos `.dbc` são convertidos automaticamente para `.parquet` e salvos em `../data/`.

In [ ]:
files = sinasc.get_files("DN", uf=["MG"], year=[2020, 2021, 2022])
files

In [ ]:
parquets = []
for f in files:
    parquet = sinasc.download([f], local_dir='../data')
    parquets.append(parquet)
    print(f"Baixado: {f}")
    time.sleep(5)

## 4. Carregamento e Merge

Lê os arquivos `.parquet` salvos, concatena os três anos em um único DataFrame e adiciona coordenadas geográficas (latitude/longitude) via merge com a tabela de municípios do IBGE.

In [ ]:
from pathlib import Path

parquets = list(Path("../data").glob("*.parquet"))

In [ ]:
df = pd.concat([pd.read_parquet(f) for f in parquets], ignore_index=True)

### Merge com coordenadas geográficas (IBGE)

O SINASC registra apenas o **código IBGE do município de residência** (`CODMUNRES`), um inteiro de 6 dígitos. Usar esse código diretamente como feature seria problemático: o modelo trataria o código como grandeza numérica ordinal (313375 > 310020), quando na verdade é apenas um identificador nominal sem ordem significativa.

A solução é converter o código em **latitude e longitude**, que:

- São variáveis contínuas com **significado espacial real** — municípios próximos têm coordenadas próximas
- Funcionam como **proxy socioeconômico**: regiões Sul/Sudeste de MG têm IDH, renda e acesso a serviços diferentes do Norte/Vale do Jequitinhonha, e o modelo pode capturar esses padrões implicitamente
- Permitem futuramente usar **features de vizinhança** (clustering geoespacial, distância a centros hospitalares de referência)

> **Detalhe técnico do join:** o código IBGE na tabela de municípios tem 7 dígitos (inclui dígito verificador), enquanto `CODMUNRES` tem 6. Por isso o merge é feito com `munic["codigo_ibge"] // 10` — dividindo por 10 para remover o último dígito e alinhar as chaves.

In [ ]:
df["CODMUNRES"] = pd.to_numeric(df["CODMUNRES"], errors="coerce")

munic = pd.read_csv(
    "https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv"
)

df = df.merge(
    munic[["codigo_ibge", "latitude", "longitude"]],
    left_on="CODMUNRES",
    right_on=munic["codigo_ibge"] // 10,
    how="left"
)

## 5. Análise Exploratória (EDA)

Visão geral do dataset: dimensões, tipos, valores nulos, estatísticas descritivas e conversão de colunas para tipos numéricos.

In [ ]:
df.shape

In [ ]:
df.describe()

### Conversão de tipos: por que todas as colunas chegam como `object`?

O SINASC distribui os dados em formato `.dbc` (dBase compactado), um formato legado do DATASUS. Ao converter para parquet via `pysus`, **todas as colunas são lidas como string (`object`)** — incluindo campos que semanticamente são numéricos, como idade, peso e semanas de gestação.

Isso acontece porque o `.dbc` não tem tipagem forte: o campo `IDADEMAE = "30"` é armazenado como texto, não como inteiro.

O bloco abaixo força a conversão das colunas que serão usadas como numéricas com `pd.to_numeric(..., errors="coerce")`:
- Valores que são de fato números (`"30"`, `"2815"`) são convertidos normalmente
- Valores inválidos ou sentinelas textuais (`""`, `" "`, `"99"` que virou texto corrompido) viram `NaN` automaticamente — o argumento `errors="coerce"` garante isso sem lançar exceção

> Colunas **não** listadas aqui (como `CODOCUPMAE`, `DTNASC`, flags administrativas) permanecem como string pois serão tratadas como categóricas ou descartadas.

In [ ]:
num_cols = [
    "IDADEMAE", "PESO", "APGAR1", "APGAR5", "QTDFILVIVO", "QTDFILMORT",
    "GESTACAO", "GRAVIDEZ", "PARTO", "CONSULTAS", "SEXO", "RACACOR",
    "IDANOMAL", "ESCMAE", "ESCMAE2010", "SEMAGESTAC", "CONSPRENAT",
    "MESPRENAT", "QTDGESTANT", "QTDPARTNOR", "QTDPARTCES", "IDADEPAI",
    "ESTCIVMAE", "LOCNASC", "PARTO"
]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

In [ ]:
df.describe()

In [ ]:
df.head()

## 6. Seleção de Features e Definição do Target

### Problema de negócio

Prever desfechos de saúde materna a partir de informações disponíveis **antes ou durante o pré-natal**, permitindo identificação precoce de risco e busca ativa em UBS.

---

### Candidatos a target (saúde materna)

| Target | Coluna SINASC | Binarização | Relevância |
|--------|--------------|-------------|------------|
| **Tipo de parto** ⭐ | `PARTO` | 1 = cesárea / 0 = vaginal | Indicador central da OMS; taxa de cesárea no Brasil é >56%, muito acima do limite recomendado de 15%. Tem relação direta com risco materno (hemorragia, infecção, tempo de recuperação) e é um problema crônico de saúde pública. |
| Prematuridade | `GESTACAO` | 1 = prematuro (<37 sem) / 0 = a termo | Associado a condições maternas como pré-eclâmpsia, infecções e insuficiência cervical. |
| Adequação do pré-natal | `CONSULTAS` | 1 = adequado (≥7 consultas) / 0 = inadequado | Reflete acesso e adesão ao cuidado. Bom proxy de vulnerabilidade social. |

> **Target escolhido: `PARTO` (tipo de parto — cesárea vs. vaginal)**
>
> É o mais interessante para o Tech Challenge: problema relevante de saúde pública com narrativa forte, classificação binária limpa, bom volume no SINASC, e as features disponíveis (idade, escolaridade, paridade, consultas, tipo de gestação) são clinicamente reconhecidas como preditores de cesárea.

---

### Features selecionadas

**Perfil da mãe** (disponível desde a 1ª consulta):
`IDADEMAE`, `ESCMAE2010`, `ESTCIVMAE`, `RACACORMAE`, `QTDGESTANT`, `QTDPARTNOR`, `QTDPARTCES`, `QTDFILVIVO`, `QTDFILMORT`

**Gestação atual** (disponível durante pré-natal):
`GRAVIDEZ` (única/dupla/tripla), `CONSPRENAT` (nº consultas realizadas), `MESPRENAT` (mês de início do pré-natal), `SEXO` (via USG)

**Geográficas** (proxy socioeconômico — ver seção 4):
`latitude`, `longitude`

**Perfil do pai:**
`IDADEPAI` — mantida apesar dos muitos nulos (~50%). O nulo aqui **não é missing aleatório**: pai ausente ou não declarado é em si um marcador de vulnerabilidade social associado a menor acesso ao pré-natal e maior risco de cesárea não planejada. A estratégia é criar uma feature `PAI_AUSENTE` (1 se nulo, 0 caso contrário) e imputar a mediana nos casos preenchidos, preservando o sinal de ausência.

---

### O que foi descartado e por quê

| Motivo | Colunas |
|--------|---------|
| **Leakage** (só disponível no parto) | `SEMAGESTAC`, `APGAR1/5`, `TPROBSON`, `TPAPRESENT`, `STTRABPART`, `STCESPARTO`, `KOTELCHUCK` |
| **Nulos demais sem sinal** (>30%, ausência não interpretável) | `DTRECORIGA`, `CODANOMAL`, `DTULTMENST`, `SERIESCMAE` |
| **Redundantes** | `ESCMAE` (→ usar `ESCMAE2010`), `CONSULTAS` (→ usar `CONSPRENAT`) |
| **Administrativas** | `ORIGEM`, `CODESTAB`, `NUMEROLOTE`, `VERSAOSIST`, `CONTADOR`, datas de sistema |

---

### Plano de tratamento

1. Strings vazias → NaN
2. Conversão de tipos (`pd.to_numeric`)
3. Sentinelas → NaN (9, 99, 999, 9999 = "ignorado" no DATASUS)
4. Filtros biológicos (idade mãe <10 ou >55)
5. Feature engineering: `PRIMIPARA`, `TEM_PERDA_FETAL`, `FAIXA_ETARIA`, `PN_TARDIO`, `PAI_AUSENTE`
6. Imputação: mediana (numéricas), "nao_informado" (categóricas)
7. One-hot encoding → split 80/20 estratificado → StandardScaler

In [ ]:
cols_prenatal = [
    "PESO",  # target
    "PARTO",
    # Perfil da mãe (sabe desde o início)
    "IDADEMAE", "ESCMAE2010", "ESTCIVMAE", "RACACORMAE",
    "QTDGESTANT", "QTDPARTNOR", "QTDPARTCES",
    "QTDFILVIVO", "QTDFILMORT",
    # Gestação atual (sabe durante pré-natal)
    "GRAVIDEZ",        # única/dupla/tripla
    "MESPRENAT",       # quando começou pré-natal
    "CONSPRENAT",      # consultas previstas
    "SEXO",            # USG revela
    # Info pai
    "IDADEPAI",
    # Geo (proxy socioeconômico)
    "latitude", "longitude",
]

In [ ]:
df_model = df[cols_prenatal].copy()

## 7. Visualizações

Histogramas das features selecionadas, distribuição geográfica dos nascimentos em MG e matriz de correlação.

In [ ]:
df_model.shape

In [ ]:
info = pd.DataFrame({
    "dtype": df_model.dtypes,
    "non_null": df_model.notna().sum(),
    "null_pct": (df_model.isna().mean() * 100).round(1),
    "nunique": df_model.nunique(),
    "sample": df_model.iloc[0]
})
print(info.to_string())

In [ ]:
df_model.hist(bins=30, figsize=(20,15))

In [ ]:
df_model.plot(
    kind="scatter",
    x="longitude",
    y="latitude",
    figsize=(5, 3),
    alpha=0.5,
    s=2
)

In [ ]:
center = {'lat': df['latitude'].mean(), 'lon': df['longitude'].mean()}

df['cesarea'] = (df['PARTO'] == 2).astype(int)
agrupado = agrupado = df.assign(
    lat_round=df['latitude'].round(1),
    lon_round=df['longitude'].round(1)
).groupby(['lat_round', 'lon_round']).agg(
    taxa_cesarea=('cesarea', 'mean'),
    n=('cesarea', 'count')
).reset_index()

fig = px.scatter_map(agrupado, lat='lat_round', lon='lon_round',
                        color='taxa_cesarea', size=np.sqrt(agrupado['n']),
                        color_continuous_scale='Reds',
                        zoom=3, center=center)
fig.show(renderer='notebook')

In [ ]:
import numpy as np
center = {'lat': df['latitude'].mean(), 'lon': df['longitude'].mean()}

df['RISCO_PESO'] = (df['PESO'] < 2500).astype(int)
agrupado = df.assign(
    lat_round=df['latitude'].round(1),
    lon_round=df['longitude'].round(1)
).groupby(['lat_round', 'lon_round']).agg(
    risco_peso=('RISCO_PESO', 'mean'),
    n=('RISCO_PESO', 'count')
).reset_index()

fig = px.scatter_map(agrupado, lat='lat_round', lon='lon_round',
                     color='risco_peso', size=np.sqrt(agrupado['n']),
                     color_continuous_scale='Oranges',
                     range_color=[0, agrupado['risco_peso'].quantile(0.80)],
                     zoom=3, center=center)
fig.show(renderer='notebook')

In [ ]:
df.corr

In [ ]:
pd.plotting.scatter_matrix(df_model, figsize=(50,50))